In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import ast
import torch
import numpy as np
import pandas as pd
import scanpy as sc
from tqdm.auto import tqdm
from wppkg import read_json
from datasets import Dataset
from perthub.models import BiolordModel
from perthub.inference import biolord_inference

### Step 1: load `attributes_map.json` and `test_info.csv` file

`test_info.csv` involves:

- cell_type

- drug

- dose

- smiles

- rdkit2d_dose

In [3]:
attributes_map = read_json("../../data/attributes_map.json")
test_info = pd.read_csv("./test_info.csv")

# check
assert test_info["dose"].max() <= 1, "The dose values are not normalized. Check data preprocessing!"

for ct in test_info["cell_type"]:
    if ct not in attributes_map["categorical_attributes_map"]["cell_type"]:
        raise ValueError("cell_type is not in the pre-generated vocabulary. Check data preprocessing!")

# Convert rdkit2d_dose from string to list
test_info["rdkit2d_dose"] = test_info["rdkit2d_dose"].apply(ast.literal_eval)

print(f"Detected {len(test_info)} test conditions to generate!")
test_info.head()

Detected 108 test conditions to generate!


,cell_type,drug,dose,smiles,rdkit2d_dose
0,A549,Quisinostat,0.001,Cl.Cl.Cn1cc(CNCC2CCN(c3ncc(C(=O)NO)cn3)CC2)c2c...,"[-1.2930782244469885, 0.06703134710663183, 0.1..."
1,A549,Hesperadin,1.000,CCS(=O)(=O)Nc1ccc2c(c1)/C(=C(/Nc1ccc(CN3CCCCC3...,"[-0.6016143697452183, 1.1301081307824596, 1.07..."
2,A549,Flavopiridol,1.000,CN1CCC(c2c(O)cc(O)c3c(=O)cc(-c4ccccc4Cl)oc23)C...,"[-1.2930782243115633, 0.46583958458723324, 0.0..."
3,A549,Belinostat,0.100,O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO,"[1.210380195765111, -0.6170641584670467, -0.86..."
4,A549,Alvespimycin,0.001,COC1/C=C\C=C(/C)C(=O)NC2=CC(=O)C(NCCN(C)C)=C(C...,"[-1.2930782228085633, 0.7963010194093817, 1.45..."


### Step 2: add `sample_indices` and expand `test_info`

In [4]:
# Get `sample_indices` from control cells in validation dataset
adata = sc.read_h5ad("../../data/sciplex3_biolord.h5ad")
sample_indices = np.where((adata.obs["split_ood"] == "test") & (adata.obs["control"] == 1))[0]
print(f"For each test condition, we will generate {len(sample_indices)} samples!")

# Expand test_info.csv
test_info_expand = test_info.loc[test_info.index.repeat(len(sample_indices))].reset_index(drop=True)
test_info_expand["sample_indices"] = sample_indices.tolist() * len(test_info)

# Map cell_type to integer
test_info_expand["cell_type_name"] = test_info_expand["cell_type"]
test_info_expand["cell_type"] = test_info_expand["cell_type"].map(attributes_map["categorical_attributes_map"]["cell_type"])

test_info_expand.head()

For each test condition, we will generate 1128 samples!


,cell_type,drug,dose,smiles,rdkit2d_dose,sample_indices,cell_type_name
0,0,Quisinostat,0.001,Cl.Cl.Cn1cc(CNCC2CCN(c3ncc(C(=O)NO)cn3)CC2)c2c...,"[-1.2930782244469885, 0.06703134710663183, 0.1...",95,A549
1,0,Quisinostat,0.001,Cl.Cl.Cn1cc(CNCC2CCN(c3ncc(C(=O)NO)cn3)CC2)c2c...,"[-1.2930782244469885, 0.06703134710663183, 0.1...",452,A549
2,0,Quisinostat,0.001,Cl.Cl.Cn1cc(CNCC2CCN(c3ncc(C(=O)NO)cn3)CC2)c2c...,"[-1.2930782244469885, 0.06703134710663183, 0.1...",521,A549
3,0,Quisinostat,0.001,Cl.Cl.Cn1cc(CNCC2CCN(c3ncc(C(=O)NO)cn3)CC2)c2c...,"[-1.2930782244469885, 0.06703134710663183, 0.1...",1004,A549
4,0,Quisinostat,0.001,Cl.Cl.Cn1cc(CNCC2CCN(c3ncc(C(=O)NO)cn3)CC2)c2c...,"[-1.2930782244469885, 0.06703134710663183, 0.1...",2291,A549


### Step 3: construct test dataset and start inference

In [5]:
test_ds = Dataset.from_pandas(test_info_expand[["cell_type", "rdkit2d_dose", "sample_indices"]])
test_ds.set_format(type="torch")

inference_outputs = biolord_inference(
    model=BiolordModel.from_pretrained(
        "../../biolord_logs/last_model",
        ordered_attributes_map=attributes_map["ordered_attributes_map"],
        categorical_attributes_map=attributes_map["categorical_attributes_map"]
    ),
    test_ds=test_ds,
    batch_size=512,
    test_dl_num_workers=4,
    device=torch.device("cuda:0")
)

assert inference_outputs.shape[0] == test_info_expand.shape[0]

inference:   0%|          | 0/238 [00:00<?, ?it/s]

### Step 4: build AnnData with inference results

In [6]:
adata_pred = sc.AnnData(
    X=inference_outputs,
    obs=test_info_expand.set_index(test_info_expand.index.astype(str)),
    var=pd.DataFrame(index=adata.var_names.astype(str)),
)

adata_pred

AnnData object with n_obs × n_vars = 121824 × 2000
    obs: 'cell_type', 'drug', 'dose', 'smiles', 'rdkit2d_dose', 'sample_indices', 'cell_type_name'

### Step 5: calculate $r^{2}$ for each test condition

- In origin biolord repo:

    - overall_mean: 0.762390

    - overall_median: 0.852794

In [7]:
from sklearn.metrics import r2_score

# extract ground truth from OOD set
adata_ood = adata[adata.obs["split_ood"] == "ood"]

results = {}
all_test_conditions = adata_ood.obs["cov_drug_dose_name"].unique()

for cond in tqdm(all_test_conditions):
    cell_type, drug, dose_str = cond.split("_", 2)
    dose = float(dose_str)

    # ground truth: sparse matrix → mean(axis=0) → flatten to 1D array
    mask_gt = adata_ood.obs["cov_drug_dose_name"] == cond
    gt_x = np.array(adata_ood[mask_gt].X.mean(axis=0)).flatten()

    # predictions: filter adata_pred by matching condition
    mask_pre = (
        (adata_pred.obs["cell_type_name"] == cell_type)
        & (adata_pred.obs["drug"] == drug)
        & (adata_pred.obs["dose"] == dose)
    )
    pre_x = adata_pred[mask_pre].X.mean(axis=0)

    r2 = r2_score(gt_x, pre_x)
    results[cond] = r2

overall_mean = sum(results.values()) / len(results)
overall_median = np.median([v for v in results.values()]).item()
results, overall_mean, overall_median

  0%|          | 0/108 [00:00<?, ?it/s]

({'A549_Quisinostat_0.001': 0.4315118193626404,
  'A549_Hesperadin_1.0': 0.8525812029838562,
  'A549_Flavopiridol_1.0': 0.9516230821609497,
  'A549_Belinostat_0.1': 0.8455734848976135,
  'A549_Alvespimycin_0.001': 0.9665212631225586,
  'A549_Alvespimycin_0.1': 0.8588993549346924,
  'A549_TAK-901_0.1': 0.9294057488441467,
  'A549_Belinostat_1.0': 0.8586611151695251,
  'A549_Hesperadin_0.01': 0.9363303184509277,
  'A549_TAK-901_1.0': 0.9115645885467529,
  'A549_TAK-901_0.01': 0.9302542209625244,
  'A549_Quisinostat_1.0': 0.7678927779197693,
  'A549_Flavopiridol_0.01': 0.8061301708221436,
  'A549_Dacinostat_0.001': 0.9019383788108826,
  'A549_Hesperadin_0.1': 0.9157320261001587,
  'A549_Alvespimycin_1.0': 0.8463708162307739,
  'A549_Belinostat_0.001': 0.8824328780174255,
  'A549_Tanespimycin_0.01': 0.942337691783905,
  'A549_Flavopiridol_0.1': 0.943184494972229,
  'A549_Givinostat_0.1': 0.8885535597801208,
  'A549_Dacinostat_0.01': 0.7685433626174927,
  'A549_Tanespimycin_0.001': 0.975020